# Autoencoder Inference from Best Checkpoint

This notebook loads `best.pt` and runs inference on one LR TIFF image.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import tifffile
from src.lr_autoencoder.infer import load_model_from_checkpoint, infer_image

: 

In [ ]:
checkpoint_path = Path('outputs/simple_lr_autoencoder/checkpoints/best.pt')
input_image = Path('/path/to/your/test_data')

if input_image.is_dir():
    candidates = sorted(input_image.glob('*.tif')) + sorted(input_image.glob('*.tiff'))
    if not candidates:
        raise FileNotFoundError(f'No TIFF files in {input_image}')
    input_image = candidates[0]

device = 'cpu'
model, ckpt = load_model_from_checkpoint(checkpoint_path, device=device)
image_size = int(ckpt.get('data_config', {}).get('image_size', 192))
recon = infer_image(model, input_image, image_size=image_size, device=device)
raw = tifffile.imread(input_image)
print('Loaded:', input_image)
print('Reconstruction shape:', recon.shape)

In [ ]:
def as_hwc(arr):
    a = np.asarray(arr)
    if a.ndim == 2:
        return a[..., None]
    if a.ndim == 3 and a.shape[0] <= 8 and a.shape[1] > 8 and a.shape[2] > 8:
        return np.transpose(a, (1, 2, 0))
    return a

raw_hwc = as_hwc(raw)
recon_hwc = as_hwc(recon)

ch = 0
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(raw_hwc[..., ch], cmap='gray')
axes[0].set_title('Input (channel 0)')
axes[0].axis('off')
axes[1].imshow(recon_hwc[..., ch], cmap='gray')
axes[1].set_title('Reconstruction (channel 0)')
axes[1].axis('off')
plt.tight_layout()

In [ ]:
out_path = Path('outputs/inference/notebook_recon.tif')
out_path.parent.mkdir(parents=True, exist_ok=True)
tifffile.imwrite(out_path, recon.astype(np.float32))
print('Saved reconstruction to', out_path)